# 중간고사 과제 버섯 식용 및 독성 분류

22개 범주형 특성으로 식용(e)과 독성(p)을 분류한다.

## 1. 패키지 설정

In [ ]:
# 1. 패키지 설정
# 패키지가 없으면 아래 주석을 해제하여 먼저 실행한다.
# %pip install numpy pandas matplotlib seaborn scikit-learn tensorflow ipykernel

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn import metrics

RANDOM_STATE = 100

## 2. 데이터 준비

In [ ]:
Dataset = pd.read_csv('mushrooms.csv', na_values=['?', '', ' '])
print('데이터 크기:', Dataset.shape)
Dataset.head()

In [ ]:
Dataset.info()
display(Dataset.describe())

## 3. 결측치 확인 및 제거
. `?`를 결측값으로 처리하며, `n`은 정상적인 범주 코드이므로 삭제하지 않는다.

In [ ]:
NAs = pd.isna(Dataset)
sns.heatmap(NAs, yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing values before preprocessing')
plt.tight_layout()
plt.show()

print('특성별 결측값 개수')
display(Dataset.isna().sum().to_frame('missing_count'))
rows_before = len(Dataset)
Dataset = Dataset.dropna().copy()
print('제거 전:', rows_before)
print('제거한 행:', rows_before - len(Dataset))
print('제거 후:', len(Dataset))
print('남은 결측값:', Dataset.isna().sum().sum())

## 4. 데이터 탐색 EDA

In [ ]:
print(Dataset.groupby('class').size())
sns.countplot(data=Dataset, x='class', order=['e', 'p'])
plt.title('Class distribution (e=edible, p=poisonous)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

feature_names = list(Dataset.columns[1:])
print('특성 개수:', len(feature_names))
Dataset.describe()

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(18, 8))
selected_features = feature_names[0:8]
for feature, ax in zip(selected_features, axs.flat):
    sns.countplot(data=Dataset, x=feature, hue='class',
                  order=sorted(Dataset[feature].unique()), hue_order=['e', 'p'], ax=ax)
    ax.set_title(feature)
for ax in list(axs.flat)[len(selected_features):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(18, 8))
selected_features = feature_names[8:16]
for feature, ax in zip(selected_features, axs.flat):
    sns.countplot(data=Dataset, x=feature, hue='class',
                  order=sorted(Dataset[feature].unique()), hue_order=['e', 'p'], ax=ax)
    ax.set_title(feature)
for ax in list(axs.flat)[len(selected_features):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(18, 8))
selected_features = feature_names[16:22]
for feature, ax in zip(selected_features, axs.flat):
    sns.countplot(data=Dataset, x=feature, hue='class',
                  order=sorted(Dataset[feature].unique()), hue_order=['e', 'p'], ax=ax)
    ax.set_title(feature)
for ax in list(axs.flat)[len(selected_features):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

## 5. 학습용 및 테스트용 데이터 분리


In [ ]:
Dataset['class'] = Dataset['class'].apply(lambda x: 1 if x == 'p' else 0)
Y = Dataset['class'].values
X = Dataset.drop('class', axis=1)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=RANDOM_STATE, stratify=Y)

print('Train data X:', X_train.shape, ', Y:', Y_train.shape)
print('Test data X:', X_test.shape, ', Y:', Y_test.shape)

## 6. 범주형 데이터 원핫 인코딩


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                        dtype=np.float32)
X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)


print('인코딩 후 학습 데이터:', X_train_encoded.shape)
print('인코딩 후 테스트 데이터:', X_test_encoded.shape)
print(X_train_encoded[:5])

## 7. SVM 분류기


In [ ]:
from sklearn.svm import SVC
model = SVC(C=1, kernel='linear')
model.fit(X_train_encoded, Y_train)
pred = model.predict(X_test_encoded)
svm_pred = pred.copy()

In [ ]:
print('Accuracy :', metrics.accuracy_score(Y_test, pred))
print('Report :', metrics.classification_report(
    Y_test, pred, target_names=['edible (0)', 'poisonous (1)'], digits=4,
    zero_division=0))

confusion_matrix = metrics.confusion_matrix(Y_test, pred, labels=[0, 1])
cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix, display_labels=['edible', 'poisonous'])
cm_display.plot(cmap='Blues')
plt.title('SVM')
plt.show()

## 8. Decision Tree 분류기


In [ ]:
from sklearn.tree import DecisionTreeClassifier
clf_gini = DecisionTreeClassifier(criterion='gini',
                                  max_depth=3,
                                  min_samples_leaf=5,
                                  random_state=100)
clf_gini.fit(X_train_encoded, Y_train)
y_pred_gini = clf_gini.predict(X_test_encoded)
dt_pred = y_pred_gini.copy()

In [ ]:
print('Accuracy :', metrics.accuracy_score(Y_test, y_pred_gini))
print('Report :', metrics.classification_report(
    Y_test, y_pred_gini, target_names=['edible (0)', 'poisonous (1)'], digits=4,
    zero_division=0))

confusion_matrix = metrics.confusion_matrix(Y_test, y_pred_gini, labels=[0, 1])
cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix, display_labels=['edible', 'poisonous'])
cm_display.plot(cmap='Blues')
plt.title('Decision Tree')
plt.show()

In [ ]:
from sklearn import tree
plt.figure(figsize=(20, 9))
tree.plot_tree(clf_gini, feature_names=encoder.get_feature_names_out(),
               class_names=['edible', 'poisonous'], filled=True, fontsize=10)
plt.tight_layout()
plt.show()

## 9. Neural Network 분류기


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical

keras.utils.set_random_seed(3)
y_train = to_categorical(Y_train, num_classes=2)
y_test = to_categorical(Y_test, num_classes=2)

model = keras.Sequential([
    keras.Input(shape=(X_train_encoded.shape[1],)),
    keras.layers.Flatten(),
    keras.layers.Dense(20, activation='relu'),
    keras.layers.Dense(2, activation='softmax')
])
model.summary()

model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=['accuracy'])
history = model.fit(X_train_encoded, y_train,
                    validation_split=0.1, epochs=25, verbose=2)

In [ ]:
# 테스트 데이터는 학습이나 검증에 사용하지 않았다.
test_loss, test_acc = model.evaluate(X_test_encoded, y_test, verbose=0)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)
y_pred = model.predict(X_test_encoded, verbose=0)
y_pred_labels = []
for i in y_pred:
    y_pred_labels.append(np.argmax(i))
nn_pred = np.array(y_pred_labels)
print('예측값:', nn_pred[:20])
print('실제값:', Y_test[:20])

print('Accuracy :', metrics.accuracy_score(Y_test, y_pred_labels))
print('Report :', metrics.classification_report(
    Y_test, y_pred_labels, target_names=['edible (0)', 'poisonous (1)'], digits=4,
    zero_division=0))

confusion_matrix = metrics.confusion_matrix(Y_test, y_pred_labels, labels=[0, 1])
cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix, display_labels=['edible', 'poisonous'])
cm_display.plot(cmap='Blues')
plt.title('Neural Network')
plt.show()

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epochs')
plt.legend(['training data', 'validation data'], loc='lower right')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend(['training data', 'validation data'], loc='upper right')
plt.show()

## 10. 세 모델의 성능 비교
모든 모델은 동일한 테스트 데이터로 평가한다. 정밀도, 재현율, F1은 독성(p=1)을 양성 클래스로 계산한다.

In [ ]:
results = []
for name, pred in [('SVM', svm_pred), ('Decision Tree', dt_pred),
                   ('Neural Network', nn_pred)]:
    results.append({
        'Model': name,
        'Accuracy': metrics.accuracy_score(Y_test, pred),
        'Precision (poisonous)': metrics.precision_score(Y_test, pred, zero_division=0),
        'Recall (poisonous)': metrics.recall_score(Y_test, pred, zero_division=0),
        'F1 (poisonous)': metrics.f1_score(Y_test, pred, zero_division=0),
        'Poisonous predicted edible': int(((Y_test == 1) & (pred == 0)).sum())
    })
comparison = pd.DataFrame(results).set_index('Model')
display(comparison)

comparison[['Accuracy', 'Precision (poisonous)', 'Recall (poisonous)',
            'F1 (poisonous)']].plot.bar(figsize=(10, 5), rot=0)
plt.ylim(0, 1.15)
plt.ylabel('Score')
plt.title('Model comparison on the same test set')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 11. 결과 해석

결측치 2,480행을 제거한 5,644행을 사용하고 학습 4,515행, 테스트 1,129행으로 나눴다. 모델별 성능은 바로 위의 실제 실행 결과표로 확인한다. 동일한 테스트 세트를 사용했으며, 결측치가 없는 표본에 대한 평가다.
